In [4]:
import pandas as pd
import re

In [5]:
# QUESTION 1

In [6]:
num = "1024170337"
L = [int(digit) for digit in str(num) if digit.isdigit()]
last_two = L[-2:]
print(last_two)

[3, 7]


In [7]:
fixed_entries = [ 
{"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", 
"keywords": "fee cost price charge", "category": "billing"}, 
{"question": "how to reset password", "answer": "Go to Settings > Reset Password.", 
"keywords": "password reset login", "category": "account"}, 
{"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", 
"keywords": "hours timing open time", "category": "general"}, 
{"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", 
"keywords": "pay payment upi fee", "category": "billing"}, 
] 

In [8]:
categories = ["billing", "account", "general"]
custom_entries = []

In [9]:
for d in last_two:
    category = categories[d % 3]
    if category == "billing":
        custom_entries.append({"question": "how can i check my payment status",
            "answer": "Go to the Payments section to check your payment status.",
            "keywords": "payment status transaction",
            "category": "billing"})
    elif category == "account":
        custom_entries.append({ "question": "how do i update my registered mobile number",
            "answer": "Go to Account Settings to update your registered mobile number.",
            "keywords": "update mobile profile",
            "category": "account"})
    else:
        custom_entries.append({"question": "where can i find the help center",
            "answer": "You can access the Help Center from the Support section.",
            "keywords": "help support center",
            "category": "general"})
df = pd.DataFrame(fixed_entries + custom_entries)
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4            how can i check my payment status   
5  how do i update my registered mobile number   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Go to the Payments section to check your payme...   
5  Go to Account Settings to update your register...   

                     keywords category  
0       fee cost price charge  billing  
1        password reset login  account  
2      hours timing open time  general  
3         pay payment upi fee  billing  
4  payment status transaction  billing  
5       up

In [10]:
# QUESTION 2

In [13]:
def score_hypothesis(query, df):
    query_words = set(re.findall(r'\w+', query.lower()))
    results = []
    for __, row in df.iterrows():
        question_words = set(re.findall(r'\w+', row["question"].lower()))
        answer_words = set(re.findall(r'\w+', row["answer"].lower()))
        keyword_words = set(re.findall(r'\w+', row["keywords"].lower()))
        category_words = set(re.findall(r'\w+', row["category"].lower()))

        question_matches = len(query_words & question_words)
        answer_matches = len(query_words & answer_words)
        keyword_matches = len(query_words & keyword_words)
        category_matches = len(query_words & category_words)

        score = (
            question_matches * 3 +
            keyword_matches * 2 +
            answer_matches * 1 +
            category_matches * 2
        )
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "keywords": row["keywords"],
                "category": row["category"],
                "confidence": score
            })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(
            by="confidence",
            ascending=False
        ).reset_index(drop=True)

    return results_df

query = "how can i pay the fee"

matches = score_hypothesis(query, df)

print(matches)

                                      question  \
0                        how can i pay the fee   
1                       what is the annual fee   
2            how can i check my payment status   
3  how do i update my registered mobile number   
4                        how to reset password   

                                              answer  \
0         You can pay via UPI, card, or net banking.   
1                          The annual fee is Rs 500.   
2  Go to the Payments section to check your payme...   
3  Go to Account Settings to update your register...   
4                   Go to Settings > Reset Password.   

                     keywords category  confidence  
0         pay payment upi fee  billing          24  
1       fee cost price charge  billing          10  
2  payment status transaction  billing          10  
3       update mobile profile  account           6  
4        password reset login  account           3  


In [ ]:
# QUESTION 3.

In [14]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]


# Use one of the personalized categories
category_name = "billing"

print("Questions in category:", category_name)
print(same_category(category_name, df))

Questions in category: billing
0               what is the annual fee
3                how can i pay the fee
4    how can i check my payment status
Name: question, dtype: str


In [ ]:
# QUESTION 4.

In [15]:
# Pick one entry
entry_index = 4

new_keyword = input("Enter a new keyword: ")

# Add keyword to the selected entry
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

# Save the updated DataFrame
df.to_csv("_faq_data.csv", index=False)

print("\nUpdated entry:")
print(df.loc[entry_index])

print("\nDataFrame saved as _faq_data.csv")


Updated entry:
question                    how can i check my payment status
answer      Go to the Payments section to check your payme...
keywords                          payment status transaction 
category                                              billing
Name: 4, dtype: str

DataFrame saved as _faq_data.csv


In [ ]:
# QUESTION 5.

In [16]:
category_counts = df.groupby("category").size()

print("FAQ entries per category:")
print(category_counts)

FAQ entries per category:
category
account    2
billing    3
general    1
dtype: int64


In [ ]:
# QUESTION 6.

In [17]:
def score_hypothesis(query, df):
    query_words = set(re.findall(r'\w+', query.lower()))
    results = []

    for _, row in df.iterrows():

        question_words = set(
            re.findall(r'\w+', row["question"].lower())
        )

        answer_words = set(
            re.findall(r'\w+', row["answer"].lower())
        )

        keyword_words = set(
            re.findall(r'\w+', row["keywords"].lower())
        )

        category_words = set(
            re.findall(r'\w+', row["category"].lower())
        )

        question_matches = len(query_words & question_words)
        answer_matches = len(query_words & answer_words)
        keyword_matches = len(query_words & keyword_words)
        category_matches = len(query_words & category_words)

        score = (
            question_matches * 3 +
            keyword_matches * 2 +
            answer_matches +
            category_matches * 2
        )

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results_df = pd.DataFrame(results)

    if results_df.empty:
        print("No matching entries found.")
        return results_df

    # Highest score
    highest_score = results_df["confidence"].max()

    # Get ALL entries having the highest score
    best_matches = results_df[
        results_df["confidence"] == highest_score
    ].reset_index(drop=True)

    print("Best matching entries:")
    print(best_matches)

    return best_matches

In [18]:
query = "fee"
score_hypothesis(query, df)

Best matching entries:
                 question                     answer category  confidence
0  what is the annual fee  The annual fee is Rs 500.  billing           6


,question,answer,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,6


In [19]:
query = "reset password"
score_hypothesis(query, df)

Best matching entries:
                question                            answer category  \
0  how to reset password  Go to Settings > Reset Password.  account   

   confidence  
0          12  


,question,answer,category,confidence
0,how to reset password,Go to Settings > Reset Password.,account,12
